# Computer Lab: Dispersion and Chromaticity — local Python version

This notebook replaces the original Sirepo/Elegant workflow with a fully local first-order matrix model. The helper module `dc_local_helpers.py` is installed from the course GitHub repo and contains the transfer matrices, matching routines, plotting code, and widget wrappers. The notebook cells expose only the physics knobs students should inspect or modify.

The model is intentionally pedagogical: drifts, thick quadrupoles, sector bends, linear dispersion, finite-difference chromaticity, and tune-footprint plots. It is not a production lattice model.

<style>
.answer {
    color: #b00020;
    border-left: 4px solid #b00020;
    padding-left: 0.8em;
    margin: 0.8em 0;
}
.answer table, .answer th, .answer td {
    color: #b00020;
}
.instructor-note {
    color: #7a3b00;
    border-left: 4px solid #7a3b00;
    padding-left: 0.8em;
    margin: 0.8em 0;
}
.note {
    background-color: #f6f8fa;
    padding: 0.75em;
    border-left: 4px solid #999;
}
</style>


<div class="instructor-note">

**Instructor-development note.** This notebook is tagged for a single-source publishing workflow. Keep student-facing prompt, code, widget, and reflection cells tagged `student-keep`. Put worked solutions in separate red cells tagged `solution` and `solution-red`; put solution-only computation cells in cells tagged `solution-code`. The script `make_student_versions.py` removes solution and instructor cells to build the student notebook.

</div>

## Learning goals

By the end of this lab you should be able to:

1. Explain how a bend creates horizontal dispersion.
2. Use $\sigma_x^2=\epsilon_x\beta_x+\eta_x^2\sigma_\delta^2$ to estimate beam-size growth from momentum spread.
3. Tune a two-bend section to make an achromat.
4. Distinguish local zero dispersion from globally small dispersion.
5. Estimate momentum acceptance from both aperture and chromatic resonance crossing.
6. Interpret a tune footprint on a low-order resonance diagram.


In [ ]:
%load_ext autoreload
%autoreload 2

HELPER_VERSION = "main"  # Replace with a release tag, e.g. "v2026-lab1", for student releases.
HELPER_REPO = "git+https://github.com/r-hensley/uspas-2026-jupyter.git"

import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--upgrade",
    f"{HELPER_REPO}@{HELPER_VERSION}",
])

import numpy as np
import pandas as pd
import xtrack as xt
from IPython.display import HTML, display

import dc_local_helpers as dch
import plotly.io as pio

try:
    from google.colab import output as colab_output
except ImportError:
    pio.renderers.default = "plotly_mimetype"
else:
    colab_output.enable_custom_widget_manager()
    pio.renderers.default = "colab"

pd.set_option("display.precision", 6)
display(dch.check_environment())


## Model conventions

All distances are in meters. Quadrupole strength $k_1$ is in $\mathrm{m}^{-2}$. Positive $k_1$ focuses horizontally and defocuses vertically. The beam has geometric emittance

$$
\epsilon_x=\epsilon_y=6\ \mathrm{mm\,mrad}=6\times 10^{-6}\ \mathrm{m\,rad}.
$$

The visible lattice cells use modern Xsuite `Environment` syntax to name elements and assemble lines. The dispersion/chromaticity calculations are then run with a local first-order matrix model so this notebook remains independent of unavailable Sirepo/Elegant files.

The horizontal vector used internally is $[x,x',\delta]$, where $\delta=\Delta p/p_0$. Bends include first-order dispersion source terms. The aperture exercise uses an rms-envelope proxy, so the aperture condition is explicitly

$$
n_\sigma\sigma_x = R.
$$

The default is $n_\sigma=1$ to match the simplified style of the original lab. You can change it later.


In [31]:
emit = dch.GEOMETRIC_EMITTANCE
sigma_delta = 1e-3       # 0.1% momentum spread
pipe_radius_m = 0.025    # 2.5 cm
n_sigma_aperture = 1.0   # use 1 for the original simplified rms criterion


# A. Dispersion in a FODO lattice

We start from a 5 m FODO cell. The reference case has no bend. The modified case replaces the 0.5 m bend slot with a 20-degree sector bend. This is the local-code replacement for editing the FODO lattice in Sirepo.

In the quadrupole-only transport labs, every particle with the same initial transverse coordinates follows the same first-order transverse optics regardless of momentum. A dipole adds a new effect: **horizontal dispersion**, usually written $\eta_x$ or $D_x$. Dispersion describes how the closed-orbit position changes for particles whose momentum differs from the reference momentum.

The physical reason is magnetic rigidity. An off-momentum particle has a different

$$
B\rho = \frac{p}{q},
$$

so the same dipole field bends it by a slightly different amount. Just like $\beta$ and $\alpha$, the dispersion can have a matched periodic solution in a repeating cell.


## A0. Reference matched FODO cell with no dipole

Run the baseline matched cell. Record the beam sizes at QF and QD. These are your comparison values before adding a dipole.

The cell below builds the FODO lattice in the same style used in current Xsuite examples: create an `xt.Environment`, define named elements with `env.new(...)`, and assemble an `xt.Line` from a list of component names.

In [32]:
env_fodo = xt.Environment()
env_fodo["kq"] = 0.6

env_fodo.new("D1", xt.Drift, length=1.0)
env_fodo.new("QF", xt.Quadrupole, length=0.5, k1="kq")
env_fodo.new("D2", xt.Drift, length=2.0)
env_fodo.new("QD", xt.Quadrupole, length=0.5, k1="-kq")
env_fodo.new("D3", xt.Drift, length=1.0)

fodo_reference_line = env_fodo.new_line(
    name="fodo_reference",
    components=["D1", "QF", "D2", "QD", "D3"],
)
fodo_reference_line.particle_ref = xt.Particles(p0c=1e9, mass0=xt.ELECTRON_MASS_EV, q0=-1)

fodo_reference = dch.elements_from_xsuite_line(
    fodo_reference_line,
    roles={
        "QF": "focusing quadrupole",
        "QD": "defocusing quadrupole",
    },
)
optics_fodo_reference = dch.compute_periodic_optics(fodo_reference)

display(dch.element_layout(fodo_reference))
display(dch.optics_summary(optics_fodo_reference, label="reference FODO"))
display(dch.table_at_element_centers(optics_fodo_reference, ["QF", "QD"], sigma_delta=0.0))

,name,kind,s_start_m,s_end_m,length_m,k1_m^-2,angle_deg,edge_angle_deg,role
0,D1,drift,0.0,1.0,1.0,NaN,NaN,NaN,
1,QF,quad,1.0,1.5,0.5,0.6,NaN,NaN,focusing quadrupole
2,D2,drift,1.5,3.5,2.0,NaN,NaN,NaN,
3,QD,quad,3.5,4.0,0.5,-0.6,NaN,NaN,defocusing quadrupole
4,D3,drift,4.0,5.0,1.0,NaN,NaN,NaN,


,quantity,reference FODO,min,max
0,length [m],5.0,NaN,NaN
1,stable x?,True,NaN,NaN
2,stable y?,True,NaN,NaN
3,cell tune Qₓ,0.113502,NaN,NaN
4,cell tune Qᵧ,0.113502,NaN,NaN
5,βₓ [m],NaN,5.031928,10.189479
6,βᵧ [m],NaN,5.031928,10.189479
7,ηₓ [m],NaN,0.000000,0.000000


,element,s_center_m,beta_x_m,beta_y_m,eta_x_m,eta_xp,sigma_x_mm,sigma_y_mm
0,QF,1.25,10.189479,5.031928,0.0,0.0,7.819007,5.494685
1,QD,3.75,5.031928,10.189479,0.0,0.0,5.494685,7.819007


In [33]:
dch.plot_optics(optics_fodo_reference, title="Reference matched FODO cell: no bend")
dch.plot_beam_size(optics_fodo_reference, sigma_delta=0.0, title="Reference FODO beam size, σδ = 0")


## A1. Replace part of the central drift with a dipole

Now insert a 20-degree bend. The default model below uses a sector bend without edge focusing, so the main new effect is horizontal dispersion. This makes the lattice edit visible: the 2.0 m central drift from the reference FODO is split into a 1.5 m drift plus a 0.5 m Xsuite `xt.Bend`.

In [34]:
env_bend = xt.Environment()
env_bend["kq"] = env_fodo["kq"]
env_bend["bend_angle"] = np.deg2rad(20.0)
env_bend["bend_length"] = 0.5

env_bend.new("D1", xt.Drift, length=1.0)
env_bend.new("QF", xt.Quadrupole, length=0.5, k1="kq")
env_bend.new("D2", xt.Drift, length=1.5)
env_bend.new(
    "BEND",
    xt.Bend,
    length="bend_length",
    angle="bend_angle",
    k0="bend_angle / bend_length",
)
env_bend.new("QD", xt.Quadrupole, length=0.5, k1="-kq")
env_bend.new("D3", xt.Drift, length=1.0)

fodo_with_bend_line = env_bend.new_line(
    name="fodo_with_bend",
    components=["D1", "QF", "D2", "BEND", "QD", "D3"],
)
fodo_with_bend_line.particle_ref = xt.Particles(p0c=1e9, mass0=xt.ELECTRON_MASS_EV, q0=-1)

fodo_with_bend = dch.elements_from_xsuite_line(
    fodo_with_bend_line,
    roles={
        "QF": "focusing quadrupole",
        "BEND": "20-degree bend slot",
        "QD": "defocusing quadrupole",
    },
)
optics_fodo_bend = dch.compute_periodic_optics(fodo_with_bend)

display(dch.element_layout(fodo_with_bend))
display(dch.optics_summary(optics_fodo_bend, label="FODO with 20-degree bend"))
dch.plot_optics(optics_fodo_bend, title="FODO cell with a 20-degree bend: beta functions and dispersion")

,name,kind,s_start_m,s_end_m,length_m,k1_m^-2,angle_deg,edge_angle_deg,role
0,D1,drift,0.0,1.0,1.0,NaN,NaN,NaN,
1,QF,quad,1.0,1.5,0.5,0.6,NaN,NaN,focusing quadrupole
2,D2,drift,1.5,3.0,1.5,NaN,NaN,NaN,
3,BEND,bend,3.0,3.5,0.5,NaN,20.0,0.0,20-degree bend slot
4,QD,quad,3.5,4.0,0.5,-0.6,NaN,NaN,defocusing quadrupole
5,D3,drift,4.0,5.0,1.0,NaN,NaN,NaN,


,quantity,FODO with 20-degree bend,min,max
0,length [m],5.0,NaN,NaN
1,stable x?,True,NaN,NaN
2,stable y?,True,NaN,NaN
3,cell tune Qₓ,0.199401,NaN,NaN
4,cell tune Qᵧ,0.113502,NaN,NaN
5,βₓ [m],NaN,3.014292,5.389291
6,βᵧ [m],NaN,5.031928,10.189479
7,ηₓ [m],NaN,0.894851,1.315690


### Optional comparison: add bend edge focusing

The default FODO-with-bend model above uses a sector bend **without** edge focusing. That choice isolates the main new effect of the bend: horizontal dispersion. In the original Sirepo/Elegant lab, the rectangular bend included edge focusing, so the dipole also changed the betatron optics even when the beam had no momentum spread.

#### A comment on dipole edge focusing

A real dipole field does not stop abruptly at the magnet face. Fringe fields extend beyond the entrance and exit, and a tilted pole face gives the beam a thin-lens kick. This is called **edge focusing**.

Edge focusing is separate from dispersion. It does **not** require an energy spread in the beam. If you track a distribution with $\sigma_\delta=0$, dispersion does not add beam-size growth, but edge focusing can still change $\beta_x$, $\beta_y$, and therefore the rms beam sizes.

In this local model, the slider below applies equal entrance and exit face angles to `BEND`. Positive edge angle gives horizontal defocusing and vertical focusing kicks at the bend faces. Watch the summary table and optics plot: changes that appear at $\sigma_\delta=0$ are optics changes, not dispersive beam-size growth.

The full Xsuite `Environment` setup was shown above. From this point on, `dch.make_fodo_cell(...)` rebuilds that same FODO cell inside a helper so we do not repeat the whole lattice setup every time. This cell focuses only on the physics change: use the slider to choose the bend-edge angle, then select the bend by name and assign its entrance and exit face angles.


In [65]:
from IPython.display import HTML
from ipywidgets import FloatSlider, interactive_output, VBox

edge_angle_slider = FloatSlider(
    value=10.0,
    min=0.0,
    max=15.0,
    step=1.0,
    description="edge angle [deg]",
    continuous_update=False,
)


def update_edge_focusing(edge_angle_deg):
    edge_angle = np.deg2rad(edge_angle_deg)

    fodo_with_edges = dch.make_fodo_cell(kq=0.6, with_bend=True, bend_angle_deg=20.0)
    fodo_with_edges["BEND"].edge_entry_angle = edge_angle
    fodo_with_edges["BEND"].edge_exit_angle = edge_angle

    try:
        optics_fodo_edges = dch.compute_periodic_optics(fodo_with_edges)
    except ValueError as exc:
        print(exc)
        display(dch.stability_report(fodo_with_edges))
        return

    display(dch.element_layout(fodo_with_edges))

    def summary_for_comparison(result, label):
        return dch.optics_summary(result, label).set_index("quantity").rename(
            columns={label: f"{label} value", "min": f"{label} min", "max": f"{label} max"}
        )

    comparison = pd.concat(
        [
            summary_for_comparison(optics_fodo_reference, "no bend"),
            summary_for_comparison(optics_fodo_bend, "bend, no edge focusing"),
            summary_for_comparison(optics_fodo_edges, "bend + edge focusing"),
        ],
        axis=1,
    ).reset_index()
    display(comparison)
    fig = dch.plot_optics(
        optics_fodo_edges,
        title=f"FODO cell with 20-degree bend and {edge_angle_deg:.1f}-degree edge angles",
        show=False,
    )

    if pio.renderers.default == "colab":
        display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))
    else:
        fig.show()


edge_angle_output = interactive_output(update_edge_focusing, {"edge_angle_deg": edge_angle_slider})
display(VBox([edge_angle_output, edge_angle_slider]))

#### Effect of dispersion on beam size

After inserting the dipole, inspect the optics plot and focus on the horizontal dispersion $\eta_x$. Its scale is usually much smaller than the $\beta$ functions, so in an interactive Plotly figure it can help to hide $\beta_x$ and $\beta_y$ from the legend and look at $\eta_x$ by itself.

Dispersion is not automatically a beam-size increase. The horizontal rms size is

$$
\sigma_x^2 = \epsilon_x\beta_x + \eta_x^2\sigma_\delta^2,
$$

where $\sigma_\delta=\Delta p/p_0$ is the rms fractional momentum spread. If $\sigma_\delta=0$, the dispersive term is zero. If $\sigma_\delta$ is nonzero, locations with larger $|\eta_x|$ are more sensitive to momentum spread.

In this model there is no vertical dispersion, so $\sigma_y$ does not get an $\eta_y\sigma_\delta$ contribution. Vertical beam-size changes come from changes in the vertical optics, for example from quadrupoles or from optional bend edge focusing.


## Q1. Minimum dispersion and beam-size comparison

Find the minimum horizontal dispersion in the FODO cell with the bend. At what element or element region does it occur: focusing quadrupole, defocusing quadrupole, drift, or bend?

Then compare the no-bend and bend cases. Which beam-size differences come from dispersion, and which differences can occur even at $\sigma_\delta=0$ because the bend changed the optics?


In [36]:
display(dch.dispersion_extrema(optics_fodo_bend))

display(dch.beam_size_comparison_at_elements(optics_fodo_reference, ["QF", "QD"], sigma_delta=0.0))
display(dch.beam_size_comparison_at_elements(optics_fodo_bend, ["QF", "QD"], sigma_delta=0.0))


,condition,s_m,element,kind,eta_x_m,eta_xp,beta_x_m,beta_y_m
0,minimum eta_x,3.6875,QD,quad,0.894851,0.014347,3.124260,10.165999
1,maximum eta_x,1.2500,QF,quad,1.315690,-0.012153,5.384278,5.031928
2,maximum |eta_x|,1.2500,QF,quad,1.315690,-0.012153,5.384278,5.031928


,element,"sigma_x, delta=0 [mm]","sigma_y, delta=0 [mm]"
0,QF,7.819007,5.494685
1,QD,5.494685,7.819007


,element,"sigma_x, delta=0 [mm]","sigma_y, delta=0 [mm]"
0,QF,5.683808,5.494685
1,QD,4.292659,7.819007


**Your Q1 answer**

Minimum $\eta_x$ =  
Location =  
Beam-size comparison with the no-bend FODO =  
Which differences are dispersive? Which are optical/edge-focusing differences?


<div class="answer">

**Q1 answer.** In the bent FODO cell, the minimum horizontal dispersion is

$$
\eta_{x,\min}=0.894851\ \mathrm{m}
$$

at $s=3.6875\ \mathrm{m}$, inside **QD**. The maximum dispersion is $1.315690\ \mathrm{m}$, near the **QF** center.

With $\sigma_\delta=0$, the no-bend reference gives approximately

| element | $\sigma_x$ [mm] | $\sigma_y$ [mm] |
|---|---:|---:|
| QF | 7.819 | 5.495 |
| QD | 5.495 | 7.819 |

The bent cell gives approximately

| element | $\sigma_x$ [mm] | $\sigma_y$ [mm] |
|---|---:|---:|
| QF | 5.684 | 5.495 |
| QD | 4.293 | 7.819 |

Because $\sigma_\delta=0$, these beam-size differences are **not** due to the dispersive term $(\eta_x\sigma_\delta)^2$. They come from the changed horizontal optics of the sector bend. In this default model edge focusing is off, so the vertical optics are unchanged from the no-bend FODO reference.

</div>


## Q2. Beam size with 0.1% momentum spread

Assume

$$
\sigma_\delta=\frac{\Delta p}{p_0}=0.001.
$$

At QF, compute the expected horizontal beam size from

$$
\sigma_x = \sqrt{\epsilon_x\beta_x + (\eta_x\sigma_\delta)^2}.
$$

Compare with the $\sigma_\delta=0$ result. Then do the same comparison for $\sigma_y$.


In [37]:
sigma_delta = 1e-3

display(dch.table_at_element_centers(optics_fodo_bend, ["QF"], sigma_delta=sigma_delta))
display(dch.beam_size_comparison_at_elements(optics_fodo_bend, ["QF"], sigma_delta=sigma_delta))

dch.plot_beam_size(optics_fodo_bend, sigma_delta=sigma_delta, title="FODO with bend: effect of 0.1% momentum spread")


,element,s_center_m,beta_x_m,beta_y_m,eta_x_m,eta_xp,sigma_x_mm,sigma_y_mm
0,QF,1.25,5.384278,5.031928,1.31569,-0.012153,5.834099,5.494685


,element,"sigma_x, delta=0 [mm]","sigma_x, delta=0.001 [mm]","sigma_y, delta=0 [mm]","sigma_y, delta=0.001 [mm]"
0,QF,5.683808,5.834099,5.494685,5.494685


**Your Q2 answer**

| quantity at QF | $\sigma_\delta=0$ | $\sigma_\delta=0.001$ |
|---|---:|---:|
| $\sigma_x$ | | |
| $\sigma_y$ | | |

Explain why the horizontal and vertical sizes behave differently.


<div class="answer">

**Q2 answer.** At QF, for $\sigma_\delta=0.001$, the table gives

| quantity at QF | $\sigma_\delta=0$ | $\sigma_\delta=0.001$ |
|---|---:|---:|
| $\sigma_x$ | 5.684 mm | 5.834 mm |
| $\sigma_y$ | 5.495 mm | 5.495 mm |

The horizontal size increases because

$$
\sigma_x=\sqrt{\beta_x\epsilon_x+(\eta_x\sigma_\delta)^2}.
$$

The vertical size is unchanged in this model because there is no vertical dispersion term and the momentum spread was only applied to the horizontal dispersive contribution.

</div>


## Try it: bend angle and momentum spread

Move the bend-angle and $\sigma_\delta$ sliders. Watch how $\eta_x$, $\sigma_x$, and $\sigma_y$ respond. This replaces the old Sirepo workflow of editing the lattice and rerunning with a new `Sigma DP` value.


In [38]:
dch.interactive_fodo_dispersion()


interactive(children=(FloatSlider(value=0.6, description='quad k1', max=1.2, min=0.1, step=0.05), FloatSlider(…

<function dc_local_helpers.interactive_fodo_dispersion.<locals>.update(kq=0.6, bend_angle_deg=20.0, sigma_delta=0.001, edge_focusing=False)>

# B. Designing a zero-dispersion insert

The next model is a compact two-bend achromat-like section with two 18-degree bends. With all quadrupoles off, a beam that starts with zero dispersion does not end with zero dispersion. We will use quadrupoles to make $\eta_x$ and $\eta_x'$ return to zero.


## B0. Two-bend cell with all quadrupoles off

Use transport optics, not periodic optics, for this first part. The initial conditions are

$$
\beta_x=\beta_y=10\ \mathrm{m},\qquad \alpha_x=\alpha_y=0,
$$

with zero incoming dispersion. Again, the lattice is assembled with Xsuite `Environment` syntax before being converted to the local first-order model used for the achromat exercise.

In [39]:
env_dba_off = xt.Environment()
env_dba_off["bend_angle"] = np.deg2rad(18.0)
env_dba_off["bend_length"] = 1.0

env_dba_off.new("D0", xt.Drift, length=0.5)
env_dba_off.new("Q2", xt.Quadrupole, length=0.3, k1=0.0)
env_dba_off.new("D1", xt.Drift, length=0.5)
env_dba_off.new("B1", xt.Bend, length="bend_length", angle="bend_angle", k0="bend_angle / bend_length")
env_dba_off.new("D2", xt.Drift, length=2.3)
env_dba_off.new("Q1", xt.Quadrupole, length=0.3, k1=0.0)
env_dba_off.new("D3", xt.Drift, length=2.3)
env_dba_off.new("B2", xt.Bend, length="bend_length", angle="bend_angle", k0="bend_angle / bend_length")
env_dba_off.new("D4", xt.Drift, length=0.5)
env_dba_off.new("Q3", xt.Quadrupole, length=0.3, k1=0.0)
env_dba_off.new("D5", xt.Drift, length=0.5)

two_bend_off_line = env_dba_off.new_line(
    name="two_bend_off",
    components=["D0", "Q2", "D1", "B1", "D2", "Q1", "D3", "B2", "D4", "Q3", "D5"],
)
two_bend_off_line.particle_ref = xt.Particles(p0c=1e9, mass0=xt.ELECTRON_MASS_EV, q0=-1)

two_bend_off = dch.elements_from_xsuite_line(
    two_bend_off_line,
    roles={
        "Q2": "upstream flanking quadrupole",
        "B1": "first DBA bend",
        "Q1": "central dispersion-control quadrupole",
        "B2": "second DBA bend",
        "Q3": "downstream flanking quadrupole",
    },
)
optics_two_bend_off = dch.compute_transport_optics(two_bend_off)

display(dch.element_layout(two_bend_off))
eta_end_off, etap_end_off = dch.endpoint_dispersion(two_bend_off)
display(pd.DataFrame({"quantity": ["end eta_x [m]", "end eta_x'"], "value": [eta_end_off, etap_end_off]}))
dch.plot_optics(optics_two_bend_off, title="Two-bend cell with all quadrupoles off")

,name,kind,s_start_m,s_end_m,length_m,k1_m^-2,angle_deg,edge_angle_deg,role
0,D0,drift,0.0,0.5,0.5,NaN,NaN,NaN,
1,Q2,quad,0.5,0.8,0.3,0.0,NaN,NaN,upstream flanking quadrupole
2,D1,drift,0.8,1.3,0.5,NaN,NaN,NaN,
3,B1,bend,1.3,2.3,1.0,NaN,18.0,0.0,first DBA bend
4,D2,drift,2.3,4.6,2.3,NaN,NaN,NaN,
5,Q1,quad,4.6,4.9,0.3,0.0,NaN,NaN,central dispersion-control quadrupole
6,D3,drift,4.9,7.2,2.3,NaN,NaN,NaN,
7,B2,bend,7.2,8.2,1.0,NaN,18.0,0.0,second DBA bend
8,D4,drift,8.2,8.7,0.5,NaN,NaN,NaN,
9,Q3,quad,8.7,9.0,0.3,0.0,NaN,NaN,downstream flanking quadrupole


,quantity,value
0,end eta_x [m],2.621015
1,end eta_x',0.440788


## Q3. End-of-cell dispersion before correction

Record $\eta_x$ and $\eta_x'$ at the end of the uncorrected cell. Does this cell qualify as achromatic at its exit?


In [40]:
eta_end, etap_end = dch.endpoint_dispersion(two_bend_off)
pd.DataFrame({"quantity": ["end eta_x [m]", "end eta_x'"], "value": [eta_end, etap_end]})


,quantity,value
0,end eta_x [m],2.621015
1,end eta_x',0.440788


**Your Q3 answer**

$\eta_x(s_\mathrm{end})$ =  
$\eta_x'(s_\mathrm{end})$ =  
Achromatic at the end? Why or why not?


<div class="answer">

**Q3 answer.** With both bends on and all quadrupoles off, the outgoing dispersion is

$$
\eta_x(s_\mathrm{end})=2.6210\ \mathrm{m}, \qquad
\eta'_x(s_\mathrm{end})=0.4408.
$$

This is **not** achromatic at the end. An achromatic insert requires both $\eta_x$ and $\eta'_x$ to return to zero at the exit, assuming zero incoming dispersion.

</div>


## B1. Scan Q1 to cancel outgoing dispersion

The central quadrupole Q1 changes the dispersion trajectory between the two bends. First inspect the scan. Then use the optimizer result as a check.


In [41]:
q1_scan = dch.scan_q1_for_achromat(qmin=0.0, qmax=6.0, n=301)
display(q1_scan.sort_values("penalty").head(10))
dch.plot_q1_scan(q1_scan, title="Manual scan: endpoint dispersion versus Q1")


,q1_m^-2,eta_x_end_m,eta_xp_end,penalty
117,2.34,-0.004733,-0.000795,0.000023
116,2.32,0.016949,0.002848,0.000295
118,2.36,-0.026402,-0.004436,0.000717
115,2.30,0.038644,0.006493,0.001536
119,2.38,-0.048058,-0.008075,0.002375
114,2.28,0.060352,0.010141,0.003745
120,2.40,-0.069701,-0.011711,0.004995
113,2.26,0.082073,0.013791,0.006926
121,2.42,-0.091331,-0.015345,0.008577
112,2.24,0.103807,0.017443,0.011080


In [42]:
q1_achromat = dch.best_q1_for_achromat(qmin=0.0, qmax=6.0)

display(dch.dba_endpoint_table(q1=q1_achromat, q2=0.0, q3=0.0))
print(f"Best Q1 from local scan/optimizer: {q1_achromat:.6f} 1/m^2")


,quantity,value
0,eta_x at end [m],-1.006111e-07
1,eta_x' at end,-1.690519e-08
2,penalty eta_x^2 + eta_x'^2,1.040837e-14


Best Q1 from local scan/optimizer: 2.335633 1/m^2


## Q4. Q1 strength for approximately zero outgoing dispersion

Report the Q1 strength you found, with at least two decimal places.

**Your Q4 answer**

$k_{1,\mathrm{Q1}} =$


<div class="answer">

**Q4 answer.** The refined scan gives

$$
k_{1,\mathrm{Q1}} = 2.335633\ \mathrm{m}^{-2},
$$

so reporting $k_{1,\mathrm{Q1}}\approx 2.34\ \mathrm{m}^{-2}$ is sufficient. At this value, the computed endpoint values are approximately

$$
\eta_x(s_\mathrm{end})=-1.0\times 10^{-7}\ \mathrm{m},\qquad
\eta'_x(s_\mathrm{end})=-1.7\times 10^{-8},
$$

which is zero to the precision of this simple numerical search.

</div>


## Try it: tune the achromat quadrupole by hand

Move Q1 and watch the final values of $\eta_x$ and $\eta_x'$. The best setting is where both endpoint values are near zero.


In [43]:
dch.interactive_achromat_q1()


interactive(children=(FloatSlider(value=2.3356332610219, description='Q1 k1', max=6.0, step=0.05), Output()), …

<function dc_local_helpers.interactive_achromat_q1.<locals>.update(q1=2.3356332610219)>

## B2. Add focusing so the cell can be repeated

Canceling outgoing dispersion in an open beamline does not automatically make a stable periodic cell. The default focused DBA-like cell keeps the same Q1 achromat condition and adds two flanking quadrupoles Q2 and Q3. Their placement is outside the zero-dispersion insert, so they can change periodic focusing while preserving the local achromat condition.


In [44]:
central_q1_cell = dch.make_dba_cell(q1=q1_achromat, q2=0.0, q3=0.0)
focused_dba_cell = dch.make_dba_cell(q1=q1_achromat, q2=dch.DBA_Q2_DEFAULT, q3=dch.DBA_Q3_DEFAULT)

print("Central-Q1-only cell stability:")
display(dch.stability_report(central_q1_cell))

print("Focused DBA-like cell stability:")
display(dch.stability_report(focused_dba_cell))

display(dch.dba_endpoint_table(q1=q1_achromat, q2=dch.DBA_Q2_DEFAULT, q3=dch.DBA_Q3_DEFAULT))

optics_focused_dba = dch.compute_periodic_optics(focused_dba_cell)
display(dch.optics_summary(optics_focused_dba, label="focused DBA cell"))
dch.plot_optics(optics_focused_dba, title="Stable focused DBA-like periodic cell")


Central-Q1-only cell stability:


,plane,trace(M)/2,stable?,cell tune if stable,matched beta at start [m]
0,x,-1.870030,False,NaN,NaN
1,y,4.444247,False,NaN,NaN
2,dispersion source,NaN,n/a,-1.006111e-07,-1.690519e-08


Focused DBA-like cell stability:


,plane,trace(M)/2,stable?,cell tune if stable,matched beta at start [m]
0,x,-0.832453,True,5.934679e-01,8.025767e+00
1,y,-0.062721,True,2.599888e-01,2.385471e+01
2,dispersion source,NaN,n/a,-1.391698e-07,-7.656750e-08


,quantity,value
0,eta_x at end [m],-1.391698e-07
1,eta_x' at end,-7.656750e-08
2,penalty eta_x^2 + eta_x'^2,2.523081e-14


,quantity,focused DBA cell,min,max
0,length [m],9.5,NaN,NaN
1,stable x?,True,NaN,NaN
2,stable y?,True,NaN,NaN
3,cell tune Qₓ,0.593468,NaN,NaN
4,cell tune Qᵧ,0.259989,NaN,NaN
5,βₓ [m],NaN,4.188476e-01,12.825116
6,βᵧ [m],NaN,2.204438e+00,44.079337
7,ηₓ [m],NaN,-7.409217e-08,0.888874


## Q5. Maximum dispersion in the focused DBA-like cell

A dispersion-free insert can still have a dispersion bump elsewhere. Report the maximum $\eta_x$ in the stable focused cell and its location.


In [45]:
display(dch.dispersion_extrema(optics_focused_dba))
display(dch.table_at_element_centers(optics_focused_dba, ["Q1", "B1", "B2", "Q2", "Q3"], sigma_delta=0.0))


,condition,s_m,element,kind,eta_x_m,eta_xp,beta_x_m,beta_y_m
0,minimum eta_x,0.68,Q2,quad,-7.409217e-08,-1.260989e-09,12.824451,9.239115
1,maximum eta_x,4.78,Q1,quad,8.888743e-01,-6.232621e-02,6.380086,2.405366
2,maximum |eta_x|,4.78,Q1,quad,8.888743e-01,-6.232621e-02,6.380086,2.405366


,element,s_center_m,beta_x_m,beta_y_m,eta_x_m,eta_xp,sigma_x_mm,sigma_y_mm
0,Q1,4.75,6.380086,2.405366,8.888743e-01,-6.232621e-02,6.187125,3.798973
1,B1,1.80,4.647057,5.581032,3.918918e-02,1.564345e-01,5.280373,5.786725
2,B2,7.70,4.647056,27.706419,3.918921e-02,-1.564345e-01,5.280373,12.893352
3,Q2,0.65,12.825116,9.867853,-7.368679e-08,-1.224173e-08,8.772155,7.694616
4,Q3,8.85,4.299130,43.432054,-3.363974e-08,-2.467262e-08,5.078856,16.142872


**Your Q5 answer**

Maximum $\eta_x$ =  
Location =  
Why this does not contradict “zero-dispersion insert”:


<div class="answer">

**Q5 answer.** In the focused DBA-like cell, the maximum horizontal dispersion is approximately

$$
\eta_{x,\max}=0.888874\ \mathrm{m}
$$

near $s=4.78\ \mathrm{m}$, inside **Q1**. This does not contradict the phrase “zero-dispersion insert.” In this exercise, “zero-dispersion” means the dispersion and its slope are zero at the **exit** of the insert for zero incoming dispersion. The dispersion can still form a large internal bump between the entrance and exit.

</div>


## Momentum acceptance limited by dispersion

Because the effect of dispersion depends on momentum spread, it can limit how much momentum spread can be transported without losing particles. A particle distribution that fits comfortably on momentum can become too wide horizontally when $|\eta_x|\sigma_\delta$ is large.

With a finite pipe radius, momentum spread can make the horizontal envelope too large. The simplified criterion used here is

$$
n_\sigma\sqrt{\epsilon_x\beta_x+\eta_x^2\sigma_\delta^2}=R.
$$

This condition is checked along the cell. The limiting location is not always just the point with the largest $\eta_x$, because the betatron contribution $\epsilon_x\beta_x$ also matters. The aperture limit comes from the place where the combined betatron-plus-dispersive envelope reaches the pipe first.

The original lab effectively used the $n_\sigma=1$ rms proxy. You can later change `n_sigma_aperture` to test a stricter engineering convention.


## Q6. Momentum-spread limit from a 2.5 cm pipe radius

Using $R=2.5\ \mathrm{cm}$ and $n_\sigma=1$, find the largest $\sigma_\delta$ that can be transported before the horizontal rms envelope reaches the pipe radius.


In [46]:
aperture_limits = dch.aperture_limit_table(
    optics_focused_dba,
    pipe_radius_m=pipe_radius_m,
    n_sigma=n_sigma_aperture,
)
display(aperture_limits.head(10))

dispersion_limit = float(aperture_limits.iloc[0]["max_sigma_delta"])
dispersion_limit_location = str(aperture_limits.iloc[0]["element"])
print(f"Dispersion/aperture limit: sigma_delta = {dispersion_limit:.6f} ({100*dispersion_limit:.3f}%)")
print(f"Limiting element: {dispersion_limit_location}")


,s_m,element,kind,beta_x_m,eta_x_m,sigma_x_beta_mm,pipe_radius_mm,n_sigma,max_sigma_delta,max_delta_percent
0,4.780000,Q1,quad,6.380086,0.888874,6.187125,25.0,1.0,0.027251,2.725053
1,4.720000,Q1,quad,6.186228,0.888874,6.092402,25.0,1.0,0.027278,2.727753
2,4.840000,Q1,quad,6.469667,0.881406,6.230409,25.0,1.0,0.027469,2.746885
3,4.660000,Q1,quad,5.894593,0.881406,5.947063,25.0,1.0,0.027550,2.754957
4,4.900000,Q1,quad,6.451965,0.866531,6.221880,25.0,1.0,0.027943,2.794290
5,4.600000,D2,drift,5.514963,0.866531,5.752371,25.0,1.0,0.028077,2.807655
6,4.962162,D3,drift,6.378755,0.847322,6.186480,25.0,1.0,0.028587,2.858708
7,4.537838,D2,drift,5.089500,0.847322,5.526029,25.0,1.0,0.028775,2.877491
8,5.024324,D3,drift,6.307168,0.828113,6.151667,25.0,1.0,0.029261,2.926090
9,4.475676,D2,drift,4.682573,0.828113,5.300513,25.0,1.0,0.029503,2.950278


Dispersion/aperture limit: sigma_delta = 0.027251 (2.725%)
Limiting element: Q1


**Your Q6 answer**

Largest tolerable $\sigma_\delta$ =  
Equivalent percent momentum spread =  
Limiting element/location =


<div class="answer">

**Q6 answer.** For the default $R=2.5\ \mathrm{cm}$ pipe radius and the notebook's 1-rms envelope convention, the limiting momentum spread is

$$
\sigma_\delta = 0.0272505,
$$

or about

$$
2.725\%.
$$

The limiting location is the high-dispersion region inside **Q1**, near $s=4.78\ \mathrm{m}$, where $\eta_x\approx 0.889\ \mathrm{m}$. This result depends directly on the chosen aperture criterion; using a 2-rms or 3-rms envelope would give a smaller allowed momentum spread.

</div>


## Q7. Where does beam loss occur first?

If the momentum spread exceeds your Q6 value, identify where the horizontal envelope first reaches or exceeds the aperture. Is it simply the place with the largest $\eta_x$, or does $\beta_x$ also matter?


In [47]:
sigma_delta_test = 1.10 * dispersion_limit

dch.plot_beam_size_with_aperture(
    optics_focused_dba,
    sigma_delta=sigma_delta_test,
    pipe_radius_m=pipe_radius_m,
    n_sigma=n_sigma_aperture,
)

display(dch.aperture_summary(optics_focused_dba, pipe_radius_m=pipe_radius_m, n_sigma=n_sigma_aperture))


,quantity,value
0,limiting momentum spread,0.027251
1,limiting momentum spread [%],2.725053
2,limiting element,Q1
3,limiting s [m],4.78
4,eta_x at limit [m],0.888874
5,beta_x at limit [m],6.380086


**Your Q7 answer**

Loss/worst-envelope location =  
Reason =


<div class="answer">

**Q7 answer.** For the plotted test case $\sigma_\delta=1.10$ times the Q6 limit, the first sampled points that exceed the aperture occur just upstream of Q1, in **D2** around $s\approx4.48\ \mathrm{m}$. The worst envelope is inside **Q1**, around $s\approx4.78\ \mathrm{m}$, where $\sigma_x\approx27.35\ \mathrm{mm}$, exceeding the 25 mm pipe radius.

The reason is the combination of large $\eta_x$ and nonzero betatron beam size. The wall is reached where the quadrature sum $\sqrt{\beta_x\epsilon_x+(\eta_x\sigma_\delta)^2}$ is largest relative to the available aperture.

</div>


## Try it: aperture and momentum spread

Use the sliders to compare different momentum spreads and different aperture criteria. In particular, compare $n_\sigma=1$, 3, and 5.


In [48]:
dch.interactive_aperture(optics_focused_dba)


interactive(children=(FloatSlider(value=0.001, description='sigma_delta', max=0.04, readout_format='.3f', step…

<function dc_local_helpers.interactive_aperture.<locals>.update(sigma_delta=0.001, pipe_radius_cm=2.5, n_sigma=1.0)>

# C. Chromaticity in a ring

Now repeat the stable DBA-like cell 10 times to form a model ring. A ring has betatron tunes $Q_x$ and $Q_y$. Off-momentum particles see different effective focusing, so their tunes shift with $\delta$. This derivative is chromaticity.

The DBA-like cell has regions where the dispersion is locally close to zero. That is useful: off-momentum particles do not get a large dispersive beam-size contribution in those regions. But zero dispersion in part of the lattice does **not** mean the ring has no chromaticity.

Chromaticity comes from the momentum dependence of focusing. An off-momentum particle is not bent and focused exactly like the synchronous, on-energy particle, so it oscillates with a slightly different tune. In this section we first calculate the on-energy tunes, then calculate the tune spread produced by a finite momentum spread.


In [49]:
n_ring_cells = 10
ring_cell = focused_dba_cell
ring = dch.repeat_cell(ring_cell, n_cells=n_ring_cells)

ring_info = dch.ring_summary(ring_cell, n_cells=n_ring_cells)
display(ring_info)

dch.plot_optics(optics_focused_dba, title="One cell of the repeated DBA-like ring")


,quantity,value
0,number of cells,1.000000e+01
1,Qx ring tune,5.934679e+00
2,Qy ring tune,2.599888e+00
3,Cx = dQx/d(delta),-1.018276e+01
4,Cy = dQy/d(delta),-1.506135e+01
5,max eta_x in one cell [m],8.888743e-01
6,min eta_x in one cell [m],-7.409217e-08


## Q8. Record the x and y tunes

Record $Q_x$ and $Q_y$ to three significant figures.


In [50]:
Qx, Qy = dch.ring_tunes(ring_cell, n_cells=n_ring_cells)
print(f"Qx = {Qx:.6f}")
print(f"Qy = {Qy:.6f}")


Qx = 5.934679
Qy = 2.599888


**Your Q8 answer**

$Q_x=$  
$Q_y=$


<div class="answer">

**Q8 answer.** For the 10-cell model ring,

$$
Q_x = 5.934679, \qquad Q_y = 2.599888.
$$

To three significant figures, report

$$
Q_x\approx5.93, \qquad Q_y\approx2.60.
$$

</div>


## Q9. Why are the tunes not necessarily equal?

In the earlier no-bend FODO reference case, the horizontal and vertical phase advances were equal because the lattice was symmetric between the two planes. Inspect the beta functions in the DBA-like cell and explain why equal tunes should not be expected here.


In [51]:
dch.plot_optics(optics_focused_dba, title="Beta functions and dispersion in one DBA-like cell")
display(dch.optics_summary(optics_focused_dba, label="one DBA-like cell"))


,quantity,one DBA-like cell,min,max
0,length [m],9.5,NaN,NaN
1,stable x?,True,NaN,NaN
2,stable y?,True,NaN,NaN
3,cell tune Qₓ,0.593468,NaN,NaN
4,cell tune Qᵧ,0.259989,NaN,NaN
5,βₓ [m],NaN,4.188476e-01,12.825116
6,βᵧ [m],NaN,2.204438e+00,44.079337
7,ηₓ [m],NaN,-7.409217e-08,0.888874


**Your Q9 answer**

Reason the horizontal and vertical tunes need not be equal:


<div class="answer">

**Q9 answer.** The horizontal and vertical tunes need not be equal because the horizontal and vertical one-cell transfer maps are not symmetric copies of each other. The bends affect the horizontal plane through curvature and dispersion, while the vertical plane sees different focusing. The added Q2 and Q3 strengths also break the simple symmetric FODO relationship that made the no-bend reference cell look more balanced.

</div>


## Q10. Tune spread for 0.1% momentum spread

Use

$$
\Delta Q_x=C_x\sigma_\delta,\qquad \Delta Q_y=C_y\sigma_\delta.
$$

For this question use $\sigma_\delta=0.001$.


In [52]:
Cx, Cy = dch.chromaticity_finite_difference(ring_cell, n_cells=n_ring_cells)
chromatic_table = dch.chromatic_spread_table(Qx, Qy, Cx, Cy, sigma_delta=1e-3)
display(chromatic_table)


,quantity,value
0,Qx,5.934679
1,Qy,2.599888
2,Cx,-10.182759
3,Cy,-15.061355
4,Delta Qx for sigma_delta=0.001,-0.010183
5,Delta Qy for sigma_delta=0.001,-0.015061


**Your Q10 answer**

$C_x=$  
$C_y=$  
$\Delta Q_x$ for $\sigma_\delta=0.001$ =  
$\Delta Q_y$ for $\sigma_\delta=0.001$ =


<div class="answer">

**Q10 answer.** The finite-difference chromaticities are

$$
C_x = -10.182759, \qquad C_y = -15.061355.
$$

For $\sigma_\delta=0.001$, the tune shifts across one rms momentum spread are

$$
\Delta Q_x = C_x\sigma_\delta = -0.010183,
$$

$$
\Delta Q_y = C_y\sigma_\delta = -0.015061.
$$

The signs mean that particles with positive momentum offset move to lower tune in this fixed-magnet model; for footprint-width estimates, the magnitudes are the important numbers.

</div>


## Resonance diagram and chromatic tune footprint

A resonance line satisfies

$$
mQ_x+nQ_y=p,
$$

where $m$, $n$, and $p$ are integers. The resonance order is $|m|+|n|$. In this exercise, assume resonances of order 3 and below are forbidden, while order 4 and above are tolerable.


In [53]:
# Start with the 0.1% momentum spread used in Q10.
dch.plot_tune_footprint(Qx, Qy, Cx, Cy, sigma_delta=1e-3, resonance_order=3)


## Try it: move the chromatic tune footprint

Increase $\sigma_\delta$ until the tune-footprint line first crosses a resonance of order 3 or lower. Compare your visual estimate with the table in Q11.


In [54]:
dch.interactive_tune_footprint(Qx, Qy, Cx, Cy)


interactive(children=(FloatSlider(value=0.001, description='sigma_delta', max=0.01, readout_format='.4f', step…

<function dc_local_helpers.interactive_tune_footprint.<locals>.update(sigma_delta=0.001, resonance_order=3)>

## Q11. Momentum acceptance from chromatic resonance crossing

Using resonances through order 3, estimate the largest $\sigma_\delta$ that can be tolerated before the chromatic tune footprint hits a forbidden resonance. Report the value to the nearest 0.1% if using the graph, and compare to the algorithmic value in the table.


In [55]:
resonance_limits = dch.first_resonance_crossing(Qx, Qy, Cx, Cy, max_order=3)
display(resonance_limits.head(10))

chromaticity_limit = float(resonance_limits.iloc[0]["abs_delta_at_crossing"])
nearest_resonance = str(resonance_limits.iloc[0]["resonance"])
print(f"Chromaticity/resonance limit: sigma_delta = {chromaticity_limit:.6f} ({100*chromaticity_limit:.3f}%)")
print(f"Nearest forbidden resonance: {nearest_resonance}")


,m,n,p,order,delta_at_crossing,abs_delta_at_crossing,delta_percent,resonance
0,1,2,11,3,0.003336,0.003336,0.333591,1 Qx + 2 Qy = 11
1,0,3,8,3,-0.004434,0.004434,0.443374,0 Qx + 3 Qy = 8
2,1,0,6,1,-0.006415,0.006415,0.641490,1 Qx + 0 Qy = 6
3,2,0,12,2,-0.006415,0.006415,0.641490,2 Qx + 0 Qy = 12
4,3,0,18,3,-0.006415,0.006415,0.641490,3 Qx + 0 Qy = 18
5,0,2,5,2,0.006632,0.006632,0.663210,0 Qx + 2 Qy = 5
6,2,1,14,3,0.013245,0.013245,1.324548,2 Qx + 1 Qy = 14
7,1,-2,1,3,0.013295,0.013295,1.329483,1 Qx + -2 Qy = 1
8,2,1,15,3,-0.014982,0.014982,1.498168,2 Qx + 1 Qy = 15
9,0,3,7,3,0.017698,0.017698,1.769795,0 Qx + 3 Qy = 7


Chromaticity/resonance limit: sigma_delta = 0.003336 (0.334%)
Nearest forbidden resonance: 1 Qx + 2 Qy = 11


**Your Q11 answer**

Largest tolerable $\sigma_\delta$ from the resonance diagram =  
Nearest forbidden resonance =


<div class="answer">

**Q11 answer.** The first order-3-or-lower resonance crossing found by the algorithm is at

$$
\sigma_\delta = 0.003336,
$$

or about

$$
0.3336\%.
$$

The nearest resonance is

$$
Q_x + 2Q_y = 11.
$$

A graphical estimate from the footprint plot should be close to $0.3\%$, depending on how finely the slider is adjusted.

</div>


## Q12. Compare dispersion and chromaticity limits

Compare the aperture/dispersion limit from Q6 with the chromaticity/resonance limit from Q11. The smaller value sets the momentum acceptance in this simplified model.


In [56]:
comparison = dch.acceptance_comparison(
    dispersion_limit_delta=dispersion_limit,
    chromatic_limit_delta=chromaticity_limit,
)
display(comparison)


,limitation mechanism,momentum spread limit,limit [%],note
0,dispersion/aperture,0.027251,2.725053,n_sigma * sigma_x reaches pipe radius
1,chromaticity/resonance,0.003336,0.333591,footprint reaches order <= 3 resonance
2,overall limiting mechanism,0.003336,0.333591,chromaticity/resonance


**Your Q12 answer**

The stricter momentum-acceptance limit is:  
Reason:


<div class="answer">

**Q12 answer.** The chromaticity/resonance limit is stricter in this simplified model:

| mechanism | limit in $\sigma_\delta$ | limit [%] |
|---|---:|---:|
| dispersion/aperture | 0.027251 | 2.725 |
| chromaticity/resonance | 0.003336 | 0.3336 |

Thus the overall momentum acceptance is set by the chromatic tune footprint crossing the $Q_x+2Q_y=11$ resonance before the dispersive beam size reaches the 2.5 cm aperture.

</div>


# Optional extension exercises

1. Repeat Q6 with `n_sigma_aperture = 3`. Does the aperture limit become more or less restrictive than the chromaticity limit?
2. Change the FODO bend angle in Section A. How does $\eta_x$ scale with bend angle for small angles?
3. Change Q2 and Q3 in the DBA cell using `dch.interactive_dba_stability()`. Find a stable cell with smaller maximum $\beta_y$.
4. Change the number of ring cells. What happens to total tune and chromaticity?
5. Add edge focusing in the FODO or DBA builders. Which conclusions change, and which remain the same?
